# Kalman Fair Value for Market Making

This notebook implements the one-state Kalman model from Section 3.5.

The hidden state is the latent fair value of one asset. The latest trade updates that fair value, and trade size controls the measurement noise: larger trades receive more trust.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


In [ ]:
rng = np.random.default_rng(11)
n = 300
true_fair = 100 + np.cumsum(rng.normal(0, 0.08, n))
trade_size = rng.lognormal(mean=3.1, sigma=0.7, size=n)
trade_size = np.clip(trade_size, 5, 120)
observation_noise = rng.normal(0, 0.9 / np.sqrt(trade_size / trade_size.max()), n)
trade_price = true_fair + observation_noise

trades = pd.DataFrame(
    {
        "true_fair_value": true_fair,
        "trade_price": trade_price,
        "trade_size": trade_size,
    },
    index=pd.RangeIndex(n, name="trade_id"),
)
trades.head()


In [ ]:
def kalman_fair_value(prices: pd.Series, sizes: pd.Series, state_noise: float = 0.02, base_measurement_noise: float = 0.8) -> pd.DataFrame:
    max_size = sizes.max()
    mean = float(prices.iloc[0])
    variance = 1.0
    rows = []

    for idx, price, size in zip(prices.index, prices, sizes):
        prior_mean = mean
        prior_variance = variance + state_noise
        measurement_variance = base_measurement_noise * (max_size / size)
        q = prior_variance + measurement_variance
        gain = prior_variance / q
        error = price - prior_mean
        mean = prior_mean + gain * error
        variance = (1 - gain) * prior_variance
        rows.append(
            {
                "trade_id": idx,
                "fair_value_estimate": mean,
                "prior_error": error,
                "measurement_variance": measurement_variance,
                "kalman_gain": gain,
                "state_variance": variance,
            }
        )
    return pd.DataFrame(rows).set_index("trade_id")


estimate = kalman_fair_value(trades["trade_price"], trades["trade_size"])
trades = trades.join(estimate)
trades.tail()


In [ ]:
window = 30
rolling_vwap = (
    (trades["trade_price"] * trades["trade_size"]).rolling(window).sum()
    / trades["trade_size"].rolling(window).sum()
).rename("rolling_vwap")
trades = trades.join(rolling_vwap)

error_summary = pd.DataFrame(
    {
        "kalman": (trades["fair_value_estimate"] - trades["true_fair_value"]).abs(),
        "rolling_vwap": (trades["rolling_vwap"] - trades["true_fair_value"]).abs(),
        "raw_trade": (trades["trade_price"] - trades["true_fair_value"]).abs(),
    }
).agg(["mean", "median"])
error_summary


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

trades[["true_fair_value", "trade_price", "fair_value_estimate", "rolling_vwap"]].plot(ax=axes[0], linewidth=1.5)
axes[0].set_title("Fair value estimates")

trades["trade_size"].plot(ax=axes[1], kind="bar", color="tab:gray", width=1)
axes[1].set_title("Trade size")
axes[1].set_ylabel("size")

trades["kalman_gain"].plot(ax=axes[2], color="tab:green", linewidth=2)
axes[2].set_title("Kalman gain rises when measurement noise falls")
axes[2].set_ylabel("gain")

for ax in axes:
    ax.tick_params(axis="x", labelbottom=False)
axes[2].set_xlabel("trade sequence")

plt.tight_layout();


In [ ]:
trades[["trade_size", "measurement_variance", "kalman_gain"]].corr()


## Interpretation

The market-making version uses the same trust logic as the pair-regression version. A large trade lowers measurement noise, raises the Kalman gain, and pulls the fair-value estimate closer to the latest print. A small trade does the opposite.
